In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from llama_parse import LlamaParse

parser = LlamaParse(
    result_type='markdown',
    system_prompt="Extract all board game rules. Pay special attention to 'Line of Sight', 'Action Economy', and 'Component Setup'. If there is a diagram, describe the spatial logic explicitly.",
    # this is expense. maybe start with only text information and later try visual
    # parse_mode="parse_page_with_agent",
    use_vendor_multimodal_model=True,
    vendor_multimodal_model_name="openai-gpt4o",
    vendor_multimodal_api_key=os.getenv("OPENAI_API_KEY"),
)

documents = parser.load_data('../rule_book_agent/rulebooks/Azul_230802_rules.pdf')

Started parsing the file under job_id 8f2a8017-1d68-4b35-8f86-2b43e01c7a5d


In [3]:
documents

[Document(id_='d032158e-d46a-48bf-9886-406d977a9244', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='This is the cover of the rulebook for the board game "Azul." If you need information about the rules or gameplay, feel free to ask!\n', path=None, url=None, mimetype=None), image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'),
 Document(id_='b1667c77-4b65-487d-ac43-47c9757ff731', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n### Game Setup for Azul\n\n1. **Player Boards**: \n   - Each player receives a player board (A). \n   - Flip the board to the colored side (

In [ ]:

# this didn't work because I lost the documents object from llama parse
# need to run again for experimentation because llamaparse has a quota
import pickle

with open("documents.pkl", "wb") as f:
    pickle.dump(documents, f)

NameError: name 'documents' is not defined

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import uuid

# Initialize embeddings and LLM
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")  # For generating summaries

# Create Chroma vectorstore with persistence
vectorstore = Chroma(
    collection_name='board_game_rules',
    embedding_function=embeddings,
    persist_directory='./chroma_db'
)

# Create docstore for full documents
store = InMemoryStore()
id_key = 'doc_id'

# Initialize MultiVectorRetriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key
)

# Generate summaries for each document
# MultiVectorRetriever stores summaries in vectorstore and full docs in docstore
summary_docs = []
doc_ids = []

for doc in documents:
    # Generate a concise summary of each document
    summary = llm.invoke(
        f"Summarize this board game rulebook section in 2-3 sentences, "
        f"focusing on key rules and mechanics:\n\n{doc.page_content[:2000]}"
    ).content
    
    # Create unique ID for this document
    doc_id = str(uuid.uuid4())
    doc_ids.append(doc_id)
    
    # Create summary document with metadata
    summary_doc = {
        "page_content": summary,
        "metadata": {
            **doc.metadata,
            id_key: doc_id
        }
    }
    summary_docs.append(summary_doc)

# Add summaries to vectorstore (for search)
retriever.vectorstore.add_documents(summary_docs, ids=doc_ids)

# Add full documents to docstore (for retrieval)
retriever.docstore.mset(list(zip(doc_ids, documents)))

print(f"Added {len(documents)} documents to retriever")
print(f"Vectorstore has {len(doc_ids)} summaries")
